# Lane Reroute Optimization

This notebook generates reroute suggestions for lanes with active incidents.

**Purpose:**
When a lane has an active incident (weather, equipment, traffic), this optimizer finds alternative routes and ranks them by:
- Estimated ETA improvement (negative = faster)
- Capacity fit (can the alternative handle the volume?)
- Added cost (minimize additional operational expense)

**Output:**
Reroute candidates are upserted into the `reroute_solutions` table, which the frontend uses to display rerouting options.

## Configuration

Load catalog and schema from job parameters, environment variables, or use defaults.

In [ ]:
from __future__ import annotations

import os


def _get_config(name: str, default: str) -> str:
    """Get config from job parameters (widgets), env vars, or default."""
    try:
        return dbutils.widgets.get(name)  # type: ignore[name-defined]
    except Exception:
        return os.environ.get(f"DATABRICKS_{name.upper()}", default)


CATALOG = _get_config("catalog", "main")
SCHEMA = _get_config("schema", "logistics_control_center")

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")

## Load Current State

Read the current lanes and active incidents to identify which lanes need rerouting options.

In [ ]:
lanes_df = spark.table(f"{CATALOG}.{SCHEMA}.capacity_lanes")
incidents_df = spark.table(f"{CATALOG}.{SCHEMA}.incidents").where("active = true")

lanes = [r.asDict() for r in lanes_df.collect()]
incident_lanes = {r["laneId"] for r in incidents_df.select("laneId").distinct().collect()}

print(f"Total lanes: {len(lanes)}")
print(f"Lanes with active incidents: {len(incident_lanes)}")
if incident_lanes:
    print(f"  Affected lanes: {', '.join(sorted(incident_lanes))}")

## Generate Reroute Candidates

For each lane with an active incident, find alternative lanes that:
- Match the same transport mode (air/ground)
- Have available capacity

Score each candidate based on coverage ratio, utilization, and cost factors.

In [ ]:
results: list[dict] = []

for lane in lanes:
    lane_id = lane["id"]
    if lane_id not in incident_lanes:
        continue

    demand = int(lane.get("avgDailyVolume") or 0)
    
    for candidate in lanes:
        # Skip same lane
        if candidate["id"] == lane_id:
            continue
        
        # Must match transport mode
        if candidate.get("mode") != lane.get("mode"):
            continue

        available = int(candidate.get("availableCapacity") or 0)
        if available <= 0:
            continue

        # Calculate coverage and scoring
        coverage_ratio = min(1.0, available / max(1, demand))
        utilization = float(candidate.get("utilizationPct") or 0.8)

        # ETA impact: good coverage = faster, poor coverage = slower
        if coverage_ratio >= 0.35:
            delta_eta = -int(600 + coverage_ratio * 900)  # Usually faster
        else:
            delta_eta = int(90 + (0.35 - coverage_ratio) * 600)  # Constrained

        # Cost calculation
        raw_cost = 1800 + (1.0 - coverage_ratio) * 2600 + utilization * 1000
        added_cost = round(min(5500.0, max(1500.0, raw_cost)), 2)
        capacity_used_pct = round(min(95.0, max(10.0, (demand / max(1, available)) * 100)), 2)

        strategy = f"Reroute via {candidate['origin']}->{candidate['dest']}"
        notes = (
            f"Candidate lane {candidate['id']} covers {coverage_ratio:.0%} of disrupted volume. "
            f"Mode matched on {candidate.get('mode')}."
        )
        
        results.append(
            {
                "laneId": lane_id,
                "strategy": strategy,
                "deltaETAminutes": delta_eta,
                "addedCostUSD": added_cost,
                "capacityUsedPct": capacity_used_pct,
                "notes": notes,
            }
        )

print(f"Generated {len(results)} reroute candidates")

## Upsert Reroute Solutions

Merge the new reroute candidates into the `reroute_solutions` table using a MERGE statement to update existing rows and insert new ones.

In [ ]:
if not results:
    print("No active incident lanes found. Nothing to optimize.")
else:
    out_df = spark.createDataFrame(results)
    out_df.createOrReplaceTempView("reroute_updates")
    
    spark.sql(
        f"""
        MERGE INTO {CATALOG}.{SCHEMA}.reroute_solutions AS tgt
        USING reroute_updates AS src
        ON tgt.laneId = src.laneId AND tgt.strategy = src.strategy
        WHEN MATCHED THEN UPDATE SET
          tgt.deltaETAminutes = src.deltaETAminutes,
          tgt.addedCostUSD = src.addedCostUSD,
          tgt.capacityUsedPct = src.capacityUsedPct,
          tgt.notes = src.notes
        WHEN NOT MATCHED THEN INSERT (
          laneId, strategy, deltaETAminutes, addedCostUSD, capacityUsedPct, notes
        ) VALUES (
          src.laneId, src.strategy, src.deltaETAminutes, src.addedCostUSD, src.capacityUsedPct, src.notes
        )
        """
    )
    print(f"✓ Upserted {len(results)} reroute candidates to {CATALOG}.{SCHEMA}.reroute_solutions")

## Summary

Display a sample of the generated reroute options.

In [ ]:
if results:
    print("\nSample reroute options:")
    for r in results[:5]:
        eta_str = f"{r['deltaETAminutes']:+d} min" 
        print(f"  {r['laneId']} -> {r['strategy']} | ETA: {eta_str} | Cost: ${r['addedCostUSD']:.0f}")